# 07 · Construcción del dataset definitivo del recomendador

En este notebook se juntan los datos inmobiliarios de cada barrio con los tiempos de transporte calculados en el notebook anterior.

Para ello se usan tres archivos:

* `neighborhood_real_estate_enriched.csv`, con la información inmobiliaria de los 131 barrios.
* `transit_routes_by_neighborhood.csv`, con los tiempos de ida y vuelta para cada barrio cy destino.
* `transit_route_requests_audit.csv`, con el resultado de cada consulta realizada a OpenTripPlanner.

El dataset inmobiliario tiene una fila por barrio. En cambio, el de transporte tiene una fila por cada combinación de barrio y destino. Por tanto, un mismo barrio aparece varias veces en el resultado final, pero con un destino diferente en cada fila.

También se mantienen las consultas en las que no se encontró una ruta. En estos casos no se rellenan los tiempos con medias ni con valores estimados, ya que eso supondría inventar información que OpenTripPlanner no ha calculado.

Al final se generan dos versiones principales del dataset:

1. Una versión completa, que conserva todas las combinaciones barrio–destino.
2. Una versión preparada para Pareto y KNN, que contiene únicamente las combinaciones con información completa de ida y vuelta.


In [405]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

In [406]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

In [407]:
# Número de consultas sin ruta obtenido en el notebook 06.
EXPECTED_NO_ROUTE_REQUESTS = 37


In [408]:
# Claves utilizadas en los distintos archivos.
JOIN_KEY = "zone_key"
DESTINATION_KEY = "destination_id"
SCENARIO_KEY = "scenario_id"

PAIR_KEY = [
    JOIN_KEY,
    DESTINATION_KEY,
]

REQUEST_KEY = [
    JOIN_KEY,
    DESTINATION_KEY,
    SCENARIO_KEY,
]

MORNING_SCENARIO = "morning_arrival_0900"
EVENING_SCENARIO = "evening_departure_1800"

## 1. Localización de los ficheros

El código busca automáticamente los CSV en las carpetas más habituales del proyecto. Si los tienes en otra ubicación, modifica las rutas de la siguiente celda.


In [409]:
def find_project_root(start: Path | None = None) -> Path:
    """Busca la carpeta principal del proyecto."""

    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "No se ha encontrado la carpeta principal del proyecto. "
        "Debe contener las carpetas data/ y notebooks/."
    )


In [410]:


PROJECT_ROOT = find_project_root()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FINAL_DIR = PROJECT_ROOT / "data" / "final"

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REAL_ESTATE_PATH = (
    PROCESSED_DIR
    / "neighborhood_real_estate_enriched.csv"
)

TRANSIT_PATH = (
    PROCESSED_DIR
    / "transit_routes_by_neighborhood.csv"
)

TRANSIT_AUDIT_PATH = (
    PROCESSED_DIR
    / "transit_route_requests_audit.csv"
)

FINAL_DATASET_PATH = (
    FINAL_DIR
    / "neighborhood_recommender_dataset.csv"
)

MODEL_READY_PATH = (
    FINAL_DIR
    / "neighborhood_recommender_model_ready.csv"
)

NO_ROUTE_PATH = (
    FINAL_DIR
    / "transit_no_route_requests.csv"
)

MISSING_REPORT_PATH = (
    FINAL_DIR
    / "missing_values_report.csv"
)


In [411]:

required_paths = [
    REAL_ESTATE_PATH,
    TRANSIT_PATH,
    TRANSIT_AUDIT_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    formatted_paths = "\n".join(
        f"- {path}"
        for path in missing_paths
    )

    raise FileNotFoundError(
        "Faltan los siguientes archivos:\n"
        f"{formatted_paths}"
    )


In [412]:

print("Raíz del proyecto:", PROJECT_ROOT)
print("Datos inmobiliarios:", REAL_ESTATE_PATH)
print("Resumen de transporte:", TRANSIT_PATH)
print("Auditoría de transporte:", TRANSIT_AUDIT_PATH)
print("Carpeta de salida:", FINAL_DIR)

Raíz del proyecto: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas
Datos inmobiliarios: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\neighborhood_real_estate_enriched.csv
Resumen de transporte: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\transit_routes_by_neighborhood.csv
Auditoría de transporte: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\processed\transit_route_requests_audit.csv
Carpeta de salida: C:\Users\clave\OneDrive\Documentos\ASIGNATURAS 5º INF-ADE\TFG INFO\TFG_Recomendador_Viviendas\data\final


## 2. Carga de los datasets

In [413]:
real_estate = pd.read_csv(
    REAL_ESTATE_PATH,
)

transit = pd.read_csv(
    TRANSIT_PATH,
)

transit_audit = pd.read_csv(
    TRANSIT_AUDIT_PATH,
)

In [414]:
print(
    "Dimensiones del dataset inmobiliario:",
    real_estate.shape,
)

display(real_estate.head())

Dimensiones del dataset inmobiliario: (131, 103)


,zone_key,neighborhood_name,district_name,has_real_estate_data,price_estimate_available,coverage_status,total_listings,eligible_listings,excluded_listings,eligible_percentage,excluded_percentage,market_share_percentage,num_listings,price_eur_mean,price_eur_median,price_eur_q1,price_eur_q3,price_eur_iqr,price_eur_min,price_eur_max,price_m2_mean,price_m2_median,price_m2_q1,price_m2_q3,price_m2_iqr,price_m2_std,price_m2_relative_iqr_percentage,price_m2_median_ci_low,price_m2_median_ci_high,price_m2_median_ci_width,price_m2_median_ci_relative_width_percentage,price_m2_median_bootstrap_se,bootstrap_sample_size,bootstrap_iterations,bootstrap_confidence_level,bootstrap_status,area_m2_mean,area_m2_median,area_m2_q1,area_m2_q3,area_m2_iqr,area_m2_min,area_m2_max,rooms_mean,rooms_median,rooms_known_count,rooms_unknown_count,rooms_coverage_percentage,bathrooms_mean,bathrooms_median,bathrooms_known_count,bathrooms_unknown_count,bathrooms_coverage_percentage,elevator_ratio,elevator_percentage,elevator_known_count,elevator_unknown_count,elevator_coverage_percentage,exterior_ratio,exterior_percentage,exterior_known_count,exterior_unknown_count,exterior_coverage_percentage,sample_size_score,ci_precision_score,price_homogeneity_score,price_reliability_score,price_reliability_level,price_reliability_reason,market_scope,price_type,source_snapshot_date_status,registered_price_m2_total_2025,registered_price_m2_new_2025,registered_price_m2_used_2025,registered_index_madrid_total_2025,registered_index_madrid_new_2025,registered_index_madrid_used_2025,registered_index_type_total_2025,registered_index_type_new_2025,registered_index_type_used_2025,registered_price_available,registered_new_price_available,registered_used_price_available,reference_year,price_unit,price_statistic,housing_market_scope,source_publisher,source_origin,source_series_code,minimum_cases_to_publish_neighborhood,source_coverage_2025_percentage,kaggle_asking_price_available,kaggle_features_available,kaggle_sample_reliability_score,kaggle_sample_reliability_level,price_m2_for_recommender,price_m2_for_recommender_source,housing_data_availability_status,registered_minus_kaggle_eur_m2,registered_vs_kaggle_gap_percentage,registered_to_kaggle_price_ratio
0,MAD-022,Acacias,Arganzuela,True,True,available,67,66,1,98.51,1.49,0.59,66,400937.86,399500.0,213500.00,490000.0,276500.00,108000.0,1000000.0,6121.14,6020.38,5282.58,7200.00,1917.42,1362.98,31.85,5357.14,6282.05,924.91,15.36,281.15,66,2000,0.95,calculated,71.56,63.0,35.00,86.25,51.25,12.0,280.0,2.25,2.0,52,14,78.79,1.75,2.0,12,54,18.18,0.7344,73.44,64,2,96.97,0.6562,65.62,64,2,96.97,50.79,45.24,47.22,47.68,low,La estimación debe interpretarse con precaució...,residential_sale,asking_price,unknown,5736.45,NaN,5740.58,108.53,NaN,107.63,100,NaN,100.07,True,False,True,2025,EUR_per_m2,mean_declared_transaction_price,residential_sales,Ayuntamiento de Madrid,Colegio de Registradores de España,504020100060,15,97.7,True,True,47.68,low,5736.45,registered_transactions_2025,official_price_and_kaggle_features,-283.93,-4.716147,0.952839
1,MAD-027,Atocha,Arganzuela,False,False,no_listings,0,0,0,NaN,NaN,0.00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2000,0.95,no_data,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,no_data,No hay anuncios elegibles para calcular indica...,residential_sale,asking_price,unknown,6810.36,6828.78,NaN,128.84,135.45,NaN,100,2.56,NaN,True,True,False,2025,EUR_per_m2,mean_declared_transaction_price,residential_sales,Ayuntamiento de Madrid,Colegio de Registradores de España,504020100060,15,97.7,False,False,NaN,no_data,6810.36,registered_transactions_2025,official_price_only,NaN,NaN,NaN
2,MAD-023,Chopera,Arganzuela,True,True,available,35,34,1,97.14,2.86,0.31,34,420111.47,399950.0,332500.00,509000.0,176500.00,180000.0,749900.0,5168.28,5165.79,4795.05,5528.20,733.15,652.15,14.19,4865.02,5415.78,550.76,10.66,141.68,34,2000,0.95,calculated

In [415]:
print(
    "Dimensiones del resumen de transporte:",
    transit.shape,
)

display(transit.head())

Dimensiones del resumen de transporte: (376, 30)


,zone_key,neighborhood_name,district_name,destination_id,destination_name,service_date,morning_start_datetime,morning_end_datetime,morning_transit_modes,morning_routes_used,morning_agencies_used,morning_duration_minutes,morning_walk_time_minutes,morning_waiting_time_minutes,morning_walk_distance_meters,morning_number_of_transfers,evening_start_datetime,evening_end_datetime,evening_transit_modes,evening_routes_used,evening_agencies_used,evening_duration_minutes,evening_walk_time_minutes,evening_waiting_time_minutes,evening_walk_distance_meters,evening_number_of_transfers,commute_daily_minutes,commute_daily_walk_minutes,commute_daily_waiting_minutes,commute_daily_transfers
0,MAD-011,Palacio,Centro,DEST_002,Nuevos Ministerios,2026-07-29,2026-07-29T07:51:00+02:00,2026-07-29T08:51:33+02:00,BUS,C1,Información oficial: Consorcio Regional de Tra...,60.550000,10.633333,13.000000,707.85,1.0,2026-07-29T18:00:00+02:00,2026-07-29T18:51:57+02:00,BUS,147,Información oficial: Consorcio Regional de Tra...,51.950000,21.350000,7.0,1545.31,0.0,112.500000,31.983333,20.000000,1.0
1,MAD-011,Palacio,Centro,DEST_003,UC3M Campus de Leganés,2026-07-29,2026-07-29T07:38:00+02:00,2026-07-29T08:56:51+02:00,BUS,39|482,Información oficial: Consorcio Regional de Tra...,78.850000,32.650000,11.066667,2306.94,1.0,2026-07-29T18:00:58+02:00,2026-07-29T19:23:49+02:00,BUS,493|138,Consorcio Regional de Transportes de Madrid|In...,82.850000,17.266667,12.0,1257.27,1.0,161.700000,49.916667,23.066667,2.0
2,MAD-012,Embajadores,Centro,DEST_002,Nuevos Ministerios,2026-07-29,2026-07-29T07:58:00+02:00,2026-07-29T08:49:26+02:00,BUS,27,Información oficial: Consorcio Regional de Tra...,51.433333,13.700000,5.000000,969.01,0.0,2026-07-29T18:00:00+02:00,2026-07-29T18:45:13+02:00,BUS,27,Información oficial: Consorcio Regional de Tra...,45.216667,8.633333,7.0,616.35,0.0,96.650000,22.333333,12.000000,0.0
3,MAD-012,Embajadores,Centro,DEST_003,UC3M Campus de Leganés,2026-07-29,2026-07-29T07:46:00+02:00,2026-07-29T08:56:51+02:00,BUS,E1|482,Información oficial: Consorcio Regional de Tra...,70.850000,14.050000,12.100000,1017.18,1.0,2026-07-29T18:00:58+02:00,2026-07-29T19:16:38+02:00,BUS,493|34,Consorcio Regional de Transportes de Madrid|In...,75.666667,13.533333,10.0,999.35,1.0,146.516667,27.583333,22.100000,2.0
4,MAD-013,Cortes,Centro,DEST_002,Nuevos Ministerios,2026-07-29,2026-07-29T08:06:00+02:00,2026-07-29T08:49:21+02:00,BUS,14,Información oficial: Consorcio Regional de Tra...,43.350000,12.466667,8.000000,790.59,0.0,2026-07-29T18:00:00+02:00,2026-07-29T18:35:21+02:00,BUS,27,Información oficial: Consorcio Regional de Tra...,35.350000,6.766667,7.0,492.28,0.0,78.700000,19.233333,15.000000,0.0


In [416]:
print(
    "Dimensiones de la auditoría de transporte:",
    transit_audit.shape,
)

display(transit_audit.head())

Dimensiones de la auditoría de transporte: (786, 21)


,request_key,zone_key,neighborhood_name,district_name,origin_latitude,origin_longitude,destination_id,destination_name,target_latitude,target_longitude,scenario_id,scenario_name,direction,service_date,requested_datetime,calculation_timestamp,request_status,itineraries_found,routing_errors_json,request_error,elapsed_seconds
0,MAD-011|DEST_001|morning_arrival_0900,MAD-011,Palacio,Centro,40.414971,-3.715596,DEST_001,Puerta del Sol,40.416775,-3.703790,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-28T17:55:01.899182+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,1.111223
1,MAD-011|DEST_001|evening_departure_1800,MAD-011,Palacio,Centro,40.416775,-3.703790,DEST_001,Puerta del Sol,40.414971,-3.715596,evening_departure_1800,Salida del destino a las 18:00,from_destination,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-28T17:55:03.010637+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,1.965369
2,MAD-011|DEST_002|morning_arrival_0900,MAD-011,Palacio,Centro,40.414971,-3.715596,DEST_002,Nuevos Ministerios,40.446610,-3.692436,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-28T17:55:04.976356+02:00,success,3,[],NaN,1.325776
3,MAD-011|DEST_002|evening_departure_1800,MAD-011,Palacio,Centro,40.446610,-3.692436,DEST_002,Nuevos Ministerios,40.414971,-3.715596,evening_departure_1800,Salida del destino a las 18:00,from_destination,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-28T17:55:06.302628+02:00,success,3,[],NaN,1.164491
4,MAD-011|DEST_003|morning_arrival_0900,MAD-011,Palacio,Centro,40.414971,-3.715596,DEST_003,UC3M Campus de Leganés,40.331755,-3.765697,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-28T17:55:07.467990+02:00,success,3,[],NaN,1.480304


Qué contiene cada archivo

Los tres archivos no tienen exactamente la misma estructura porque cada uno guarda un tipo de información diferente.

El dataset inmobiliario tiene una fila por barrio. El resumen de transporte tiene una fila por cada combinación de barrio y destino. Por último, la auditoría tiene dos filas por combinación, ya que se realizó una consulta para la ida de mañana y otra para la vuelta de tarde.

La auditoría es necesaria para recuperar también las consultas en las que no se encontró ninguna ruta. Estas consultas no siempre aparecen en el resumen de tiempos porque, al no existir un itinerario, no hay una duración que guardar.

## 3. Validaciones previas

In [417]:
def require_columns(
    df: pd.DataFrame,
    columns: list[str],
    df_name: str,
) -> None:
    """Comprueba que un dataset tiene las columnas necesarias."""

    missing_columns = [
        column
        for column in columns
        if column not in df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Faltan columnas en '{df_name}': "
            f"{missing_columns}"
        )

In [418]:
def validate_key_values(
    df: pd.DataFrame,
    columns: list[str],
    df_name: str,
) -> None:
    """Comprueba que las claves no tienen valores vacíos."""

    for column in columns:
        if df[column].isna().any():
            raise ValueError(
                f"Hay valores ausentes en '{column}' "
                f"dentro de '{df_name}'."
            )

        if (
            df[column]
            .astype("string")
            .str.strip()
            .eq("")
            .any()
        ):
            raise ValueError(
                f"Hay valores vacíos en '{column}' "
                f"dentro de '{df_name}'."
            )



In [419]:
require_columns(
    real_estate,
    [JOIN_KEY],
    "datos inmobiliarios",
)

require_columns(
    transit,
    PAIR_KEY,
    "resumen de transporte",
)

require_columns(
    transit_audit,
    REQUEST_KEY + ["request_status"],
    "auditoría de transporte",
)

validate_key_values(
    real_estate,
    [JOIN_KEY],
    "datos inmobiliarios",
)

validate_key_values(
    transit,
    PAIR_KEY,
    "resumen de transporte",
)

validate_key_values(
    transit_audit,
    REQUEST_KEY + ["request_status"],
    "auditoría de transporte",
)

In [420]:
# Se eliminan espacios accidentales en las claves.
real_estate[JOIN_KEY] = (
    real_estate[JOIN_KEY]
    .astype("string")
    .str.strip()
)

for column in PAIR_KEY:
    transit[column] = (
        transit[column]
        .astype("string")
        .str.strip()
    )

for column in REQUEST_KEY + ["request_status"]:
    transit_audit[column] = (
        transit_audit[column]
        .astype("string")
        .str.strip()
    )

In [421]:
# En el dataset inmobiliario debe haber una fila por barrio.
if real_estate[JOIN_KEY].duplicated().any():
    duplicated = real_estate.loc[
        real_estate[JOIN_KEY].duplicated(
            keep=False,
        )
    ]

    display(
        duplicated.sort_values(JOIN_KEY)
    )

    raise ValueError(
        "Hay barrios repetidos en el dataset inmobiliario."
    )

In [422]:
# En el resumen de transporte debe haber una fila por combinación barrio-destino.
if transit.duplicated(
    subset=PAIR_KEY
).any():
    duplicated = transit.loc[
        transit.duplicated(
            subset=PAIR_KEY,
            keep=False,
        )
    ]

    display(
        duplicated.sort_values(PAIR_KEY)
    )

    raise ValueError(
        "Hay combinaciones barrio-destino repetidas "
        "en el resumen de transporte."
    )

In [423]:
# En la auditoría debe haber una fila por consulta.
if transit_audit.duplicated(
    subset=REQUEST_KEY
).any():
    duplicated = transit_audit.loc[
        transit_audit.duplicated(
            subset=REQUEST_KEY,
            keep=False,
        )
    ]

    display(
        duplicated.sort_values(REQUEST_KEY)
    )

    raise ValueError(
        "Hay consultas de transporte repetidas "
        "en el archivo de auditoría."
    )

In [424]:
print(
    "Barrios inmobiliarios:",
    real_estate[JOIN_KEY].nunique(),
)

print(
    "Barrios en transporte:",
    transit_audit[JOIN_KEY].nunique(),
)

print(
    "Destinos:",
    transit_audit[DESTINATION_KEY].nunique(),
)

print(
    "Consultas auditadas:",
    len(transit_audit),
)

print(
    "Las claves y los posibles duplicados "
    "se han revisado correctamente."
)

Barrios inmobiliarios: 131
Barrios en transporte: 131
Destinos: 3
Consultas auditadas: 786
Las claves y los posibles duplicados se han revisado correctamente.


### Resultado de las primeras comprobaciones

Antes de unir los archivos se comprueba que las claves principales existen, no están vacías y no tienen duplicados incorrectos.

En los datos inmobiliarios, cada zone_key debe aparecer una única vez porque representa un barrio. En el resumen de transporte puede repetirse, ya que un barrio tiene una fila diferente para cada destino. En este caso, lo que no puede repetirse es la combinación de zone_key y destination_id.

En la auditoría también se tiene en cuenta el escenario de mañana o de tarde, ya que cada consulta debe aparecer una sola vez.

## 4. Recuperación del estado de las rutas

El resumen de transporte contiene los tiempos de los itinerarios encontrados, pero no es suficiente para saber qué ocurrió en todas las consultas.

Por este motivo se utiliza también el archivo de auditoría. Este archivo permite saber si cada consulta terminó correctamente o si OpenTripPlanner no encontró una ruta.

A partir de esta información se reconstruye una tabla completa con una fila por combinación barrio–destino y dos estados diferentes:

- El estado de la ruta de ida por la mañana.
- El estado de la ruta de vuelta por la tarde.

Después, cada combinación se clasifica como ruta completa, ruta parcial o sin ruta.

In [425]:
transit_audit["request_status"] = (
    transit_audit["request_status"]
    .str.lower()
)

request_status_counts = (
    transit_audit["request_status"]
    .value_counts(dropna=False)
    .rename_axis("request_status")
    .reset_index(name="consultas")
)

display(request_status_counts)

,request_status,consultas
0,success,749
1,no_route,37


In [426]:
unexpected_statuses = sorted(
    set(
        transit_audit[
            "request_status"
        ].dropna()
    )
    - {
        "success",
        "no_route",
    }
)

if unexpected_statuses:
    raise ValueError(
        "Se han encontrado estados de consulta "
        f"no esperados: {unexpected_statuses}"
    )

In [427]:
no_route_requests = (
    transit_audit.loc[
        transit_audit[
            "request_status"
        ].eq("no_route")
    ]
    .copy()
    .sort_values(REQUEST_KEY)
    .reset_index(drop=True)
)

print(
    "Consultas sin ruta:",
    len(no_route_requests),
)

Consultas sin ruta: 37


In [428]:
assert (
    len(no_route_requests)
    == EXPECTED_NO_ROUTE_REQUESTS
), (
    "Se esperaban "
    f"{EXPECTED_NO_ROUTE_REQUESTS} consultas no_route, "
    f"pero se han encontrado {len(no_route_requests)}."
)

display(
    no_route_requests.head(10)
)

,request_key,zone_key,neighborhood_name,district_name,origin_latitude,origin_longitude,destination_id,destination_name,target_latitude,target_longitude,scenario_id,scenario_name,direction,service_date,requested_datetime,calculation_timestamp,request_status,itineraries_found,routing_errors_json,request_error,elapsed_seconds
0,MAD-011|DEST_001|evening_departure_1800,MAD-011,Palacio,Centro,40.416775,-3.703790,DEST_001,Puerta del Sol,40.414971,-3.715596,evening_departure_1800,Salida del destino a las 18:00,from_destination,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-28T17:55:03.010637+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,1.965369
1,MAD-011|DEST_001|morning_arrival_0900,MAD-011,Palacio,Centro,40.414971,-3.715596,DEST_001,Puerta del Sol,40.416775,-3.703790,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-28T17:55:01.899182+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,1.111223
2,MAD-012|DEST_001|evening_departure_1800,MAD-012,Embajadores,Centro,40.416775,-3.703790,DEST_001,Puerta del Sol,40.409558,-3.701534,evening_departure_1800,Salida del destino a las 18:00,from_destination,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-28T17:59:04.353071+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,2.753140
3,MAD-012|DEST_001|morning_arrival_0900,MAD-012,Embajadores,Centro,40.409558,-3.701534,DEST_001,Puerta del Sol,40.416775,-3.703790,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-28T17:59:03.380856+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,0.972087
4,MAD-013|DEST_001|evening_departure_1800,MAD-013,Cortes,Centro,40.416775,-3.703790,DEST_001,Puerta del Sol,40.414445,-3.698547,evening_departure_1800,Salida del destino a las 18:00,from_destination,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-28T17:59:11.081610+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,2.348122
5,MAD-013|DEST_001|morning_arrival_0900,MAD-013,Cortes,Centro,40.414445,-3.698547,DEST_001,Puerta del Sol,40.416775,-3.703790,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-28T17:59:10.262715+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,0.818522
6,MAD-014|DEST_001|evening_departure_1800,MAD-014,Justicia,Centro,40.416775,-3.703790,DEST_001,Puerta del Sol,40.423895,-3.695812,evening_departure_1800,Salida del destino a las 18:00,from_destination,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-28T17:59:18.520200+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,2.314840
7,MAD-014|DEST_001|morning_arrival_0900,MAD-014,Justicia,Centro,40.423895,-3.695812,DEST_001,Puerta del Sol,40.416775,-3.703790,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-28T17:59:17.617775+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,0.902041
8,MAD-015|DEST_001|evening_departure_1800,MAD-015,Universidad,Centro,40.416775,-3.703790,DEST_001,Puerta del Sol,40.425365,-3.706674,evening_departure_1800,Salida del destino a las 18:00,from_destination,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-28T17:59:25.988118+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,1.558962
9,MAD-015|DEST_001|morning_arrival_0900,MAD-015,Universidad,Centro,40.425365,-3.706674,DEST_001,Puerta del Sol,40.416775,-3.703790,morning_arrival_0900,Llegada al destino a las 09:00,to_destination,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-28T17:59:24.617165+02:00,no_route,0,"[{""code"":""WALKING_BETTER_THAN_TRANSIT"",""descri...",NaN,1.370791


In [430]:
# Información básica de cada combinación barrio-destino.
pair_information = (
    transit_audit
    .groupby(
        PAIR_KEY,
        as_index=False,
    )
    .agg(
        neighborhood_name=(
            "neighborhood_name",
            "first",
        ),
        district_name=(
            "district_name",
            "first",
        ),
        destination_name=(
            "destination_name",
            "first",
        ),
        service_date=(
            "service_date",
            "first",
        ),
    )
)

In [431]:
# Se colocan los estados de mañana y tarde en columnas diferentes.
status_by_scenario = (
    transit_audit
    .pivot(
        index=PAIR_KEY,
        columns=SCENARIO_KEY,
        values="request_status",
    )
    .reset_index()
)

status_by_scenario.columns.name = None

status_by_scenario = (
    status_by_scenario.rename(
        columns={
            MORNING_SCENARIO:
                "morning_route_status",
            EVENING_SCENARIO:
                "evening_route_status",
        }
    )
)

required_status_columns = [
    "morning_route_status",
    "evening_route_status",
]

missing_status_columns = [
    column
    for column in required_status_columns
    if column not in status_by_scenario.columns
]

if missing_status_columns:
    raise ValueError(
        "No se han encontrado los dos escenarios "
        f"de transporte: {missing_status_columns}"
    )

In [432]:
# Se eliminan del resumen las columnas descriptivas que ya se han recuperado desde la auditoría.
duplicated_description_columns = [
    column
    for column in [
        "neighborhood_name",
        "district_name",
        "destination_name",
        "service_date",
    ]
    if column in transit.columns
]

transit_values = transit.drop(
    columns=duplicated_description_columns,
)

In [433]:
# Tabla completa con todas las combinaciones, aunque alguna no tenga ruta.
transit_complete = (
    pair_information
    .merge(
        status_by_scenario,
        on=PAIR_KEY,
        how="left",
        validate="1:1",
    )
    .merge(
        transit_values,
        on=PAIR_KEY,
        how="left",
        validate="1:1",
    )
)

transit_complete["no_route_scenarios"] = (
    transit_complete[
        [
            "morning_route_status",
            "evening_route_status",
        ]
    ]
    .eq("no_route")
    .sum(axis=1)
)

both_success = (
    transit_complete[
        "morning_route_status"
    ].eq("success")
    & transit_complete[
        "evening_route_status"
    ].eq("success")
)

both_no_route = (
    transit_complete[
        "morning_route_status"
    ].eq("no_route")
    & transit_complete[
        "evening_route_status"
    ].eq("no_route")
)

transit_complete["route_status"] = np.select(
    [
        both_success,
        both_no_route,
    ],
    [
        "complete_route",
        "no_route",
    ],
    default="partial_route",
)

expected_pairs = (
    transit_audit[JOIN_KEY].nunique()
    * transit_audit[DESTINATION_KEY].nunique()
)

assert len(transit_complete) == expected_pairs, (
    "No se han recuperado todas las combinaciones "
    "barrio-destino esperadas."
)

In [434]:
print(
    "Combinaciones barrio-destino:",
    len(transit_complete),
)

display(
    transit_complete[
        "route_status"
    ]
    .value_counts()
    .rename_axis("route_status")
    .reset_index(name="combinaciones")
)



Combinaciones barrio-destino: 393


,route_status,combinaciones
0,complete_route,373
1,no_route,17
2,partial_route,3


In [435]:
display(
    transit_complete.head()
)

,zone_key,destination_id,neighborhood_name,district_name,destination_name,service_date,evening_route_status,morning_route_status,morning_start_datetime,morning_end_datetime,morning_transit_modes,morning_routes_used,morning_agencies_used,morning_duration_minutes,morning_walk_time_minutes,morning_waiting_time_minutes,morning_walk_distance_meters,morning_number_of_transfers,evening_start_datetime,evening_end_datetime,evening_transit_modes,evening_routes_used,evening_agencies_used,evening_duration_minutes,evening_walk_time_minutes,evening_waiting_time_minutes,evening_walk_distance_meters,evening_number_of_transfers,commute_daily_minutes,commute_daily_walk_minutes,commute_daily_waiting_minutes,commute_daily_transfers,no_route_scenarios,route_status
0,MAD-011,DEST_001,Palacio,Centro,Puerta del Sol,2026-07-29,no_route,no_route,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,no_route
1,MAD-011,DEST_002,Palacio,Centro,Nuevos Ministerios,2026-07-29,success,success,2026-07-29T07:51:00+02:00,2026-07-29T08:51:33+02:00,BUS,C1,Información oficial: Consorcio Regional de Tra...,60.550000,10.633333,13.000000,707.85,1.0,2026-07-29T18:00:00+02:00,2026-07-29T18:51:57+02:00,BUS,147,Información oficial: Consorcio Regional de Tra...,51.950000,21.350000,7.0,1545.31,0.0,112.50,31.983333,20.000000,1.0,0,complete_route
2,MAD-011,DEST_003,Palacio,Centro,UC3M Campus de Leganés,2026-07-29,success,success,2026-07-29T07:38:00+02:00,2026-07-29T08:56:51+02:00,BUS,39|482,Información oficial: Consorcio Regional de Tra...,78.850000,32.650000,11.066667,2306.94,1.0,2026-07-29T18:00:58+02:00,2026-07-29T19:23:49+02:00,BUS,493|138,Consorcio Regional de Transportes de Madrid|In...,82.850000,17.266667,12.0,1257.27,1.0,161.70,49.916667,23.066667,2.0,0,complete_route
3,MAD-012,DEST_001,Embajadores,Centro,Puerta del Sol,2026-07-29,no_route,no_route,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,no_route
4,MAD-012,DEST_002,Embajadores,Centro,Nuevos Ministerios,2026-07-29,success,success,2026-07-29T07:58:00+02:00,2026-07-29T08:49:26+02:00,BUS,27,Información oficial: Consorcio Regional de Tra...,51.433333,13.700000,5.000000,969.01,0.0,2026-07-29T18:00:00+02:00,2026-07-29T18:45:13+02:00,BUS,27,Información oficial: Consorcio Regional de Tra...,45.216667,8.633333,7.0,616.35,0.0,96.65,22.333333,12.000000,0.0,0,complete_route


## 5. Comprobación de los tiempos de transporte

Cuando OpenTripPlanner no encuentra una ruta para uno de los trayectos, los campos relacionados con ese trayecto deben permanecer vacíos.

En esta parte se comprueba que las consultas no_route no tengan tiempos, distancias o transbordos informados por error.

Esta comprobación es importante porque no se van a sustituir los tiempos ausentes por medias o estimaciones. Si no existe una ruta calculada, el valor correcto es mantener el dato como ausente.

In [436]:
MORNING_ROUTE_COLUMNS = [
    "morning_duration_minutes",
    "morning_walk_time_minutes",
    "morning_waiting_time_minutes",
    "morning_walk_distance_meters",
    "morning_number_of_transfers",
]

In [437]:
EVENING_ROUTE_COLUMNS = [
    "evening_duration_minutes",
    "evening_walk_time_minutes",
    "evening_waiting_time_minutes",
    "evening_walk_distance_meters",
    "evening_number_of_transfers",
]

In [438]:
REQUIRED_MODEL_COLUMNS = [
    "commute_daily_minutes",
    "commute_daily_walk_minutes",
    "commute_daily_waiting_minutes",
    "commute_daily_transfers",
]

In [439]:
required_transport_columns = (
    MORNING_ROUTE_COLUMNS
    + EVENING_ROUTE_COLUMNS
    + REQUIRED_MODEL_COLUMNS
)

missing_transport_columns = [
    column
    for column in required_transport_columns
    if column not in transit_complete.columns
]

if missing_transport_columns:
    raise KeyError(
        "Faltan columnas necesarias para analizar "
        f"los desplazamientos: {missing_transport_columns}"
    )

In [440]:
morning_no_route = transit_complete[
    "morning_route_status"
].eq("no_route")

evening_no_route = transit_complete[
    "evening_route_status"
].eq("no_route")

invalid_morning_values = (
    transit_complete.loc[
        morning_no_route,
        MORNING_ROUTE_COLUMNS,
    ]
    .notna()
    .any(axis=1)
)

invalid_evening_values = (
    transit_complete.loc[
        evening_no_route,
        EVENING_ROUTE_COLUMNS,
    ]
    .notna()
    .any(axis=1)
)

In [441]:
print(
    "Rutas de mañana no encontradas:",
    int(morning_no_route.sum()),
)

print(
    "Rutas de tarde no encontradas:",
    int(evening_no_route.sum()),
)

print(
    "Rutas de mañana sin ruta pero con datos:",
    int(invalid_morning_values.sum()),
)

print(
    "Rutas de tarde sin ruta pero con datos:",
    int(invalid_evening_values.sum()),
)



Rutas de mañana no encontradas: 19
Rutas de tarde no encontradas: 18
Rutas de mañana sin ruta pero con datos: 0
Rutas de tarde sin ruta pero con datos: 0


In [442]:
if invalid_morning_values.any():
    display(
        transit_complete.loc[
            morning_no_route
            & invalid_morning_values,
            PAIR_KEY
            + ["morning_route_status"]
            + MORNING_ROUTE_COLUMNS,
        ]
    )

    raise ValueError(
        "Hay rutas de mañana no encontradas que tienen tiempos informados."
    )

if invalid_evening_values.any():
    display(
        transit_complete.loc[
            evening_no_route
            & invalid_evening_values,
            PAIR_KEY
            + ["evening_route_status"]
            + EVENING_ROUTE_COLUMNS,
        ]
    )

    raise ValueError(
        "Hay rutas de tarde no encontradas que tienen tiempos informados."
    )

print(
    "Los tiempos ausentes coinciden con las consultas en las que no se encontró ruta."
)

Los tiempos ausentes coinciden con las consultas en las que no se encontró ruta.


## 6. Unión de los datos inmobiliarios y de transporte

Una vez preparada la información de transporte, se une con el dataset inmobiliario mediante zone_key.

La relación es de uno a varios. Cada barrio aparece una vez en el archivo inmobiliario, pero aparece varias veces en el resultado porque se han analizado distintos destinos.

Por tanto, cada fila del dataset final representa una combinación concreta de barrio y destino. Las características inmobiliarias del barrio se repiten, mientras que los tiempos de transporte cambian según el destino.

In [ ]:
dataset = (
    real_estate
    .copy()
    .merge(
        transit_complete.copy(),
        on=JOIN_KEY,
        how="left",
        validate="1:m",
        indicator=True,
        suffixes=(
            "",
            "_transit",
        ),
    )
    .copy()
)

C:\Users\clave\AppData\Local\Temp\ipykernel_15412\1551339257.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset = real_estate.merge(


In [444]:
print(
    "Dimensiones después de la unión:",
    dataset.shape,
)

display(
    dataset["_merge"]
    .value_counts(dropna=False)
    .rename_axis("resultado_union")
    .reset_index(name="filas")
)

Dimensiones después de la unión: (393, 137)


,resultado_union,filas
0,both,393
1,left_only,0
2,right_only,0


In [445]:
unmatched_real_estate = dataset.loc[dataset["_merge"].eq("left_only")].copy()

print("Filas inmobiliarias sin coincidencia en transporte:", len(unmatched_real_estate))

if not unmatched_real_estate.empty:
    display(unmatched_real_estate[[JOIN_KEY]].drop_duplicates().sort_values(JOIN_KEY))


Filas inmobiliarias sin coincidencia en transporte: 0


### Resultado de la unión

Después de realizar la unión, todas las combinaciones de transporte deben encontrar su barrio correspondiente en el dataset inmobiliario.

El aumento del número de filas es normal. No significa que se hayan creado duplicados por error, sino que cada barrio se repite para los diferentes destinos analizados.

La columna temporal _merge permite comprobar que todas las filas tienen información procedente de ambos archivos.

## 7. Diferencia entre rutas completas e incompletas

Para utilizar una combinación barrio–destino en Pareto y KNN es necesario disponer tanto del trayecto de ida como del de vuelta.

Por ello, las combinaciones se dividen en tres grupos:

- `complete_route`: se encontraron tanto la ida de mañana como la vuelta de tarde.
- `partial_route`: solo se encontró uno de los dos trayectos.
- `no_route`: no se encontró ninguno de los dos trayectos.

Las rutas parciales y las rutas sin información se mantienen en el dataset completo, pero no pasan directamente al dataset preparado para los modelos.

De esta forma se conserva toda la información obtenida sin obligar a los algoritmos a trabajar con tiempos incompletos.

In [446]:
dataset["route_available"] = (
    dataset["route_status"]
    .eq("complete_route")
)

In [447]:
dataset["route_missing_reason"] = (
    dataset["route_status"]
    .map(
        {
            "complete_route": pd.NA,
            "partial_route":
                "falta_una_de_las_dos_rutas",
            "no_route":
                "no_hay_ruta_en_ningun_sentido",
        }
    )
    .astype("string")
)

In [448]:
dataset["usable_for_pareto_knn"] = (
    dataset["route_available"]
    & dataset[
        REQUIRED_MODEL_COLUMNS
    ].notna().all(axis=1)
)

C:\Users\clave\AppData\Local\Temp\ipykernel_15412\2117069948.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset["usable_for_pareto_knn"] = (


In [449]:
print(
    "Combinaciones con ida y vuelta:",
    int(
        dataset[
            "route_available"
        ].sum()
    ),
)

print(
    "Combinaciones utilizables para Pareto y KNN:",
    int(
        dataset[
            "usable_for_pareto_knn"
        ].sum()
    ),
)

print(
    "Combinaciones conservadas pero no utilizables:",
    int(
        (
            ~dataset[
                "usable_for_pareto_knn"
            ]
        ).sum()
    ),
)

Combinaciones con ida y vuelta: 373
Combinaciones utilizables para Pareto y KNN: 373
Combinaciones conservadas pero no utilizables: 20


In [450]:
display(
    dataset[
        "route_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("route_status")
    .reset_index(name="combinaciones")
)

,route_status,combinaciones
0,complete_route,373
1,no_route,17
2,partial_route,3


## 8. Informe de valores ausentes

In [ ]:
def missing_values_report(df: pd.DataFrame) -> pd.DataFrame:
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(2),
        "unique_non_null": df.nunique(dropna=True).values,
    })

    return report.sort_values(
        ["missing_count", "column"],
        ascending=[False, True],
    ).reset_index(drop=True)


missing_report = missing_values_report(
    dataset.drop(columns="_merge")
)

display(missing_report)


,column,dtype,missing_count,missing_pct,unique_non_null
0,route_missing_reason,string,373,94.91,2
1,registered_index_madrid_new_2025,float64,186,47.33,69
2,registered_index_type_new_2025,float64,186,47.33,69
3,registered_price_m2_new_2025,float64,186,47.33,69
4,commute_daily_minutes,float64,20,5.09,370
...,...,...,...,...,...
135,source_series_code,int64,0,0.00,1
136,source_snapshot_date_status,str,0,0.00,1
137,total_listings,int64,0,0.00,97
138,usable_for_pareto_knn,bool,0,0.00,2


In [452]:
missing_route_summary = (
    dataset
    .groupby(
        "route_status",
        dropna=False,
    )[REQUIRED_MODEL_COLUMNS]
    .agg(
        lambda series:
            int(series.isna().sum())
    )
    .T
)

missing_route_summary.index.name = "column"



In [453]:
print(
    "Valores ausentes según el estado de la ruta:"
)

display(
    missing_route_summary
)

Valores ausentes según el estado de la ruta:


route_status,complete_route,no_route,partial_route
column,,,
commute_daily_minutes,0,17,3
commute_daily_walk_minutes,0,17,3
commute_daily_waiting_minutes,0,17,3
commute_daily_transfers,0,17,3


### Lectura de los valores ausentes

No todos los valores ausentes representan un problema de calidad.

En las columnas de transporte, muchos valores vacíos se explican porque OpenTripPlanner no encontró una ruta para la mañana, para la tarde o para ninguno de los dos trayectos. En estos casos, la ausencia del dato es esperada y se mantiene de forma intencionada.

También pueden existir valores ausentes que ya procedían del dataset inmobiliario, por ejemplo porque algunos barrios no tienen suficiente información en una de las fuentes originales.

Por tanto, los valores ausentes deben interpretarse según la columna y según el estado de la ruta, en lugar de rellenarlos todos de la misma manera.

## 9. Validaciones finales

Antes de guardar los resultados se realiza una última revisión para comprobar que:

- Siguen apareciendo todos los barrios.
- Cada combinación barrio–destino aparece una sola vez.
- Se mantienen las 37 consultas `no_route`.
- No se han rellenado tiempos que no existen.
- Las filas preparadas para Pareto y KNN tienen la información necesaria.
- Las rutas parciales o incompletas no se incluyen directamente en los modelos.

Estas comprobaciones permiten detectar posibles errores en la unión antes de generar los archivos definitivos.

In [454]:
# Todas las filas deben haber encontrado información inmobiliaria y una combinación de transporte.
assert dataset["_merge"].eq("both").all(), (
    "Hay barrios que no han encontrado una "
    "combinación de transporte."
)

In [455]:
# El resultado debe tener una fila por combinación barrio-destino.
assert len(dataset) == len(transit_complete), (
    "El número de filas ha cambiado durante la unión."
)

assert dataset[PAIR_KEY].notna().all().all(), (
    "Hay combinaciones sin zone_key o destination_id."
)

assert not dataset.duplicated(
    subset=PAIR_KEY
).any(), (
    "Hay combinaciones barrio-destino repetidas."
)

In [456]:
# Todos los barrios inmobiliarios deben seguir presentes.
assert set(
    real_estate[JOIN_KEY]
) == set(
    dataset[JOIN_KEY]
), (
    "Se ha perdido algún barrio durante la unión."
)

In [457]:
# Deben mantenerse las 37 consultas sin ruta.
assert len(no_route_requests) == (
    EXPECTED_NO_ROUTE_REQUESTS
), (
    "No se han conservado las 37 consultas no_route."
)

assert int(
    dataset[
        "no_route_scenarios"
    ].sum()
) == EXPECTED_NO_ROUTE_REQUESTS, (
    "El número de escenarios no_route del dataset "
    "no coincide con la auditoría."
)

In [458]:
# Las filas consideradas utilizables deben tener todos los datos necesarios.
assert dataset.loc[
    dataset["usable_for_pareto_knn"],
    REQUIRED_MODEL_COLUMNS,
].notna().all().all(), (
    "Hay filas marcadas como utilizables "
    "con datos de transporte ausentes."
)

In [459]:
# Las rutas incompletas no pueden entrar directamente en los modelos.
assert not dataset.loc[
    dataset["route_status"].ne(
        "complete_route"
    ),
    "usable_for_pareto_knn",
].any(), (
    "Alguna ruta incompleta ha sido marcada "
    "como utilizable."
)

print(
    "Todas las validaciones finales "
    "se han superado correctamente."
)

Todas las validaciones finales se han superado correctamente.


### Conclusión de las validaciones

Las validaciones confirman que la unión mantiene todos los barrios y que cada fila representa una combinación única de barrio y destino.

También se comprueba que las 37 consultas sin ruta siguen documentadas y que sus tiempos no se han sustituido por valores artificiales.

Por tanto, el dataset conserva toda la información original, pero diferencia claramente qué filas pueden utilizarse directamente en los modelos y cuáles deben mantenerse únicamente para documentación y análisis.

## 10. Preparación de los archivos finales

Se generan cuatro archivos diferentes:

### Dataset completo

neighborhood_recommender_dataset.csv contiene todas las combinaciones barrio–destino, incluidas las rutas parciales y aquellas en las que no se encontró ningún trayecto.

Es el archivo principal y conserva toda la información del proceso.

### Dataset preparado para los modelos

neighborhood_recommender_model_ready.csv contiene únicamente las combinaciones que tienen información completa de ida y vuelta.

Este será el archivo que se utilizará directamente en los siguientes notebooks para Pareto y KNN.

### Consultas sin ruta

transit_no_route_requests.csv contiene las 37 consultas concretas en las que OpenTripPlanner no encontró un itinerario.

Se guardan por separado para poder revisar el barrio, el destino, el horario y el motivo del fallo.

### Informe de valores ausentes

missing_values_report.csv resume cuántos valores ausentes hay en cada columna del dataset completo.

In [460]:
dataset_complete = (
    dataset
    .drop(columns="_merge")
    .sort_values(PAIR_KEY)
    .reset_index(drop=True)
)

print(
    "Dataset completo:",
    dataset_complete.shape,
)

Dataset completo: (393, 139)


In [461]:
dataset_model_ready = (
    dataset_complete.loc[
        dataset_complete[
            "usable_for_pareto_knn"
        ]
    ]
    .copy()
    .sort_values(PAIR_KEY)
    .reset_index(drop=True)
)

print(
    "Dataset para Pareto y KNN:",
    dataset_model_ready.shape,
)

Dataset para Pareto y KNN: (373, 139)


In [462]:
dataset_no_route = (
    no_route_requests
    .copy()
    .sort_values(REQUEST_KEY)
    .reset_index(drop=True)
)

print(
    "Consultas no_route:",
    dataset_no_route.shape,
)



Consultas no_route: (37, 21)


In [463]:
display(
    dataset_complete.head()
)

,zone_key,neighborhood_name,district_name,has_real_estate_data,price_estimate_available,coverage_status,total_listings,eligible_listings,excluded_listings,eligible_percentage,excluded_percentage,market_share_percentage,num_listings,price_eur_mean,price_eur_median,price_eur_q1,price_eur_q3,price_eur_iqr,price_eur_min,price_eur_max,price_m2_mean,price_m2_median,price_m2_q1,price_m2_q3,price_m2_iqr,price_m2_std,price_m2_relative_iqr_percentage,price_m2_median_ci_low,price_m2_median_ci_high,price_m2_median_ci_width,price_m2_median_ci_relative_width_percentage,price_m2_median_bootstrap_se,bootstrap_sample_size,bootstrap_iterations,bootstrap_confidence_level,bootstrap_status,area_m2_mean,area_m2_median,area_m2_q1,area_m2_q3,area_m2_iqr,area_m2_min,area_m2_max,rooms_mean,rooms_median,rooms_known_count,rooms_unknown_count,rooms_coverage_percentage,bathrooms_mean,bathrooms_median,bathrooms_known_count,bathrooms_unknown_count,bathrooms_coverage_percentage,elevator_ratio,elevator_percentage,elevator_known_count,elevator_unknown_count,elevator_coverage_percentage,exterior_ratio,exterior_percentage,exterior_known_count,exterior_unknown_count,exterior_coverage_percentage,sample_size_score,ci_precision_score,price_homogeneity_score,price_reliability_score,price_reliability_level,price_reliability_reason,market_scope,price_type,source_snapshot_date_status,registered_price_m2_total_2025,registered_price_m2_new_2025,registered_price_m2_used_2025,registered_index_madrid_total_2025,registered_index_madrid_new_2025,registered_index_madrid_used_2025,registered_index_type_total_2025,registered_index_type_new_2025,registered_index_type_used_2025,registered_price_available,registered_new_price_available,registered_used_price_available,reference_year,price_unit,price_statistic,housing_market_scope,source_publisher,source_origin,source_series_code,minimum_cases_to_publish_neighborhood,source_coverage_2025_percentage,kaggle_asking_price_available,kaggle_features_available,kaggle_sample_reliability_score,kaggle_sample_reliability_level,price_m2_for_recommender,price_m2_for_recommender_source,housing_data_availability_status,registered_minus_kaggle_eur_m2,registered_vs_kaggle_gap_percentage,registered_to_kaggle_price_ratio,destination_id,neighborhood_name_transit,district_name_transit,destination_name,service_date,evening_route_status,morning_route_status,morning_start_datetime,morning_end_datetime,morning_transit_modes,morning_routes_used,morning_agencies_used,morning_duration_minutes,morning_walk_time_minutes,morning_waiting_time_minutes,morning_walk_distance_meters,morning_number_of_transfers,evening_start_datetime,evening_end_datetime,evening_transit_modes,evening_routes_used,evening_agencies_used,evening_duration_minutes,evening_walk_time_minutes,evening_waiting_time_minutes,evening_walk_distance_meters,evening_number_of_transfers,commute_daily_minutes,commute_daily_walk_minutes,commute_daily_waiting_minutes,commute_daily_transfers,no_route_scenarios,route_status,route_available,route_missing_reason,usable_for_pareto_knn
0,MAD-011,Palacio,Centro,True,True,available,308,297,11,96.43,3.57,2.67,297,771118.19,540000.0,349000.0,886000.0,537000.0,150000.0,15000000.0,7003.76,6614.32,5789.47,7480.00,1690.53,2142.05,25.56,6462.96,6898.15,435.19,6.58,113.14,297,2000,0.95,calculated,104.48,90.0,50.0,140.0,90.0,25.0,666.0,2.32,2.0,280,17,94.28,1.68,2.0,59,238,19.87,0.7162,71.62,296,1,99.66,0.7306,73.06,297,0,100.00,94.44,91.27,74.21,88.12,high,La estimación presenta una estabilidad alta en...,residential_sale,asking_price,unknown,6895.15,6421.28,6954.58,130.45,127.37,130.39,100,93.13,100.86,True,True,True,2025,EUR_per_m2,mean_declared_transaction_price,residential_sales,Ayuntamiento de Madrid,Colegio de Registradores de España,504020100060,15,97.7,True,True,88.12,high,6895.15,registered_transactions_2025,official_price_and_kaggle_features,280.83,4.245788,1.042458,DEST_001,Palacio,Centro,Puerta del Sol,2026-07-29,no_route,no_route,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [464]:
dataset_complete.to_csv(
    FINAL_DATASET_PATH,
    index=False,
    encoding="utf-8-sig",
)

dataset_model_ready.to_csv(
    MODEL_READY_PATH,
    index=False,
    encoding="utf-8-sig",
)

dataset_no_route.to_csv(
    NO_ROUTE_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_report.to_csv(
    MISSING_REPORT_PATH,
    index=False,
    encoding="utf-8-sig",
)

In [465]:
print("Archivos generados:")

for path in [
    FINAL_DATASET_PATH,
    MODEL_READY_PATH,
    NO_ROUTE_PATH,
    MISSING_REPORT_PATH,
]:
    print(
        "-",
        path.relative_to(PROJECT_ROOT),
    )

Archivos generados:
- data\final\neighborhood_recommender_dataset.csv
- data\final\neighborhood_recommender_model_ready.csv
- data\final\transit_no_route_requests.csv
- data\final\missing_values_report.csv


## 11. Resumen de los resultados

Como última comprobación se muestra un resumen con:

- El número de barrios.
- El número de destinos.
- Las combinaciones barrio–destino generadas.
- Las consultas realizadas a OpenTripPlanner.
- Las consultas correctas y las consultas sin ruta.
- Las combinaciones con ruta completa, parcial o inexistente.
- Las filas que finalmente podrán utilizarse en Pareto y KNN.

Este resumen facilita comprobar que el resultado coincide con lo obtenido en el notebook de transporte.

In [466]:
summary = pd.Series(
    {
        "barrios_inmobiliarios":
            real_estate[JOIN_KEY].nunique(),

        "destinos_analizados":
            transit_audit[
                DESTINATION_KEY
            ].nunique(),

        "combinaciones_barrio_destino":
            len(dataset_complete),

        "consultas_realizadas_a_otp":
            len(transit_audit),

        "consultas_correctas":
            int(
                transit_audit[
                    "request_status"
                ].eq("success").sum()
            ),

        "consultas_no_route":
            len(dataset_no_route),

        "combinaciones_con_ruta_completa":
            int(
                dataset_complete[
                    "route_status"
                ].eq(
                    "complete_route"
                ).sum()
            ),

        "combinaciones_con_ruta_parcial":
            int(
                dataset_complete[
                    "route_status"
                ].eq(
                    "partial_route"
                ).sum()
            ),

        "combinaciones_sin_ruta":
            int(
                dataset_complete[
                    "route_status"
                ].eq(
                    "no_route"
                ).sum()
            ),

        "filas_utilizables_para_modelos":
            len(dataset_model_ready),
    }
)

In [467]:
display(
    summary.to_frame("valor")
)

assert (
    summary["consultas_no_route"]
    == EXPECTED_NO_ROUTE_REQUESTS
)

print(
    "\nEl dataset definitivo se ha construido "
    "sin eliminar las consultas no_route "
    "ni inventar tiempos de transporte."
)

,valor
barrios_inmobiliarios,131
destinos_analizados,3
combinaciones_barrio_destino,393
consultas_realizadas_a_otp,786
consultas_correctas,749
consultas_no_route,37
combinaciones_con_ruta_completa,373
combinaciones_con_ruta_parcial,3
combinaciones_sin_ruta,17
filas_utilizables_para_modelos,373



El dataset definitivo se ha construido sin eliminar las consultas no_route ni inventar tiempos de transporte.


## 12. Conclusiones

En este notebook se han unido los datos inmobiliarios de los barrios con los resultados de transporte público obtenidos mediante OpenTripPlanner.

El resultado tiene una fila por cada combinación de barrio y destino. Esto permite comparar el mismo barrio desde diferentes necesidades de desplazamiento, ya que una zona puede estar bien comunicada con un destino y tener un tiempo mucho mayor hacia otro.

También se ha utilizado el archivo de auditoría para recuperar todas las consultas realizadas, incluidas las 37 en las que no se encontró una ruta. Estos casos se han mantenido sin rellenar sus tiempos, ya que utilizar una media o una estimación supondría añadir información que no ha sido calculada realmente.

A partir del estado de la ida y de la vuelta, las combinaciones se han separado entre rutas completas, parciales y sin ruta. Solo las rutas completas se incluyen en el archivo preparado para Pareto y KNN, aunque el resto se mantiene en el dataset completo para no perder información.

De esta forma, el dataset final permite relacionar el precio y las características de cada barrio con su accesibilidad en transporte público. Ya no se valorará una zona únicamente por ser más barata o por tener mejores viviendas, sino también por el tiempo necesario para llegar al destino seleccionado por el usuario.

### Limitaciones

Los tiempos de transporte se han calculado desde una única coordenada representativa de cada barrio. Por tanto, no reflejan las diferencias que pueden existir entre distintas calles o viviendas dentro de una misma zona.

Además, los resultados corresponden a una fecha laborable y a unos horarios concretos: llegada a las 09:00 y salida a las 18:00. Los tiempos podrían cambiar durante fines de semana, festivos o en otras franjas horarias.

Los datos proceden de los horarios planificados de transporte y no incluyen retrasos, averías, obras o incidencias reales del servicio.

Por último, una consulta `no_route` no significa necesariamente que sea imposible desplazarse desde todo el barrio. El resultado también puede estar relacionado con la coordenada utilizada, el horario elegido o con que OpenTripPlanner considere más adecuado realizar el trayecto caminando.

### Siguiente paso

El siguiente paso será utilizar el dataset preparado para construir el recomendador.

Primero se seleccionará el destino indicado por el usuario. Después se aplicarán sus preferencias de precio y vivienda y se utilizará el frente de Pareto para buscar barrios que ofrezcan un buen equilibrio entre coste y tiempo de desplazamiento.

Finalmente, KNN permitirá encontrar barrios parecidos a las opciones que mejor encajen con las preferencias introducidas.
